# 🔓 Solutions: Optimization

> **Try first, peek second.** You learn far more by struggling with a problem for ten minutes than by reading its answer for ten seconds. Use these solutions to *compare* your approach, not to replace it. There is often more than one correct way.


In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

### 🧩 Problem 8.1: Compute a cost &nbsp; ⭐

Write `mse_cost(w, b, x, y)` returning the mean squared error of the line `ŷ = w·x + b` on the data.

In [2]:
def mse_cost(w, b, x, y):
    y_hat = w * x + b
    return np.mean((y_hat - y) ** 2)

print(mse_cost(0, 0, np.array([1., 2., 3.]), np.array([2., 4., 6.])))

18.666666666666668


In [3]:
# ✔ Self-check
_x, _y = np.array([1., 2., 3.]), np.array([2., 4., 6.])
check("J(w=0, b=0) = 56/3", lambda: abs(mse_cost(0, 0, _x, _y) - 56 / 3) < 1e-9)
check("J(w=2, b=0) = 0", lambda: abs(mse_cost(2, 0, _x, _y)) < 1e-12)
check("J(w=1, b=1) = (0 + 1 + 4)/3... computed: errors −0, −1, −2", lambda: abs(mse_cost(1, 1, _x, _y) - (0 + 1 + 4) / 3) < 1e-9)

✅ PASSED   J(w=0, b=0) = 56/3
✅ PASSED   J(w=2, b=0) = 0
✅ PASSED   J(w=1, b=1) = (0 + 1 + 4)/3... computed: errors −0, −1, −2


True

**🧑‍🏫 Walkthrough.** For (w=1, b=1) the predictions are 2, 3, 4 against targets 2, 4, 6, so the errors are 0, −1, −2 and the cost is (0 + 1 + 4)/3 ≈ 1.67. Every algorithm in this lesson is built on this one function: "how bad is this choice of parameters?"

### 🧩 Problem 8.2: One gradient step by hand &nbsp; ⭐⭐

Data: `x = [1, 2, 3, 4]`, `y = [3, 5, 7, 9]` (the exact line y = 2x + 1). Start at `w = 0, b = 0` with learning rate `0.05`.

Compute the two gradients `grad_w`, `grad_b` for the MSE cost, then the updated parameters `w_new`, `b_new` after **one** step.

In [4]:
x = np.array([1., 2., 3., 4.]); y = np.array([3., 5., 7., 9.])
w, b, lr = 0.0, 0.0, 0.05

err = w * x + b - y                 # [-3, -5, -7, -9]
grad_w = 2 * np.mean(err * x)       # 2 · (−70)/4 = −35
grad_b = 2 * np.mean(err)           # 2 · (−24)/4 = −12
w_new, b_new = w - lr * grad_w, b - lr * grad_b
print(grad_w, grad_b, w_new, b_new)

-35.0 -12.0 1.75 0.6000000000000001


In [5]:
# ✔ Self-check
check("grad_w = −35", lambda: abs(grad_w + 35) < 1e-9)
check("grad_b = −12", lambda: abs(grad_b + 12) < 1e-9)
check("w_new = 1.75", lambda: abs(w_new - 1.75) < 1e-9)
check("b_new = 0.6", lambda: abs(b_new - 0.6) < 1e-9)

✅ PASSED   grad_w = −35
✅ PASSED   grad_b = −12
✅ PASSED   w_new = 1.75
✅ PASSED   b_new = 0.6


True

**🧑‍🏫 Walkthrough.** Errors are −3, −5, −7, −9; multiplying by x gives −3, −10, −21, −36 (sum −70). So grad_w = 2·(−70/4) = −35 and grad_b = 2·(−24/4) = −12. Both are negative (cost falls as the parameters rise), so the update *increases* them: w = 0 + 0.05·35 = 1.75, b = 0 + 0.05·12 = 0.6. Notice the slope parameter moved far more than the intercept: the gradient is bigger for `w` because it is multiplied by x.

### 🧩 Problem 8.3: Verify a gradient numerically &nbsp; ⭐⭐

Write `numerical_gradient(f, w, eps=1e-6)` returning the **central finite-difference** approximation `(f(w+eps) − f(w−eps)) / (2·eps)` for a function of one variable.

Use it to check the analytic gradient of `J(w) = mean((w·x − y)²)` at `w = 0.7` on the data below: store the numeric gradient in `num_grad` and the analytic one `2·mean((w·x − y)·x)` in `ana_grad`.

In [6]:
x = np.array([1., 2., 3.]); y = np.array([2., 4., 6.])
J = lambda w: np.mean((w * x - y) ** 2)

def numerical_gradient(f, w, eps=1e-6):
    return (f(w + eps) - f(w - eps)) / (2 * eps)

num_grad = numerical_gradient(J, 0.7)
ana_grad = 2 * np.mean((0.7 * x - y) * x)
print(num_grad, ana_grad)

-12.13333333360822 -12.133333333333335


In [7]:
# ✔ Self-check
check("numerical_gradient works on f(w)=w² at w=3 (→ 6)", lambda: abs(numerical_gradient(lambda w: w ** 2, 3.0) - 6) < 1e-6)
check("numeric and analytic gradients agree to 6 decimals", lambda: abs(num_grad - ana_grad) < 1e-6)
check("the analytic gradient at 0.7 is −(2/3)·(1.3·14) ≈ −16.8×… (negative)", lambda: ana_grad < 0)

✅ PASSED   numerical_gradient works on f(w)=w² at w=3 (→ 6)
✅ PASSED   numeric and analytic gradients agree to 6 decimals
✅ PASSED   the analytic gradient at 0.7 is −(2/3)·(1.3·14) ≈ −16.8×… (negative)


True

**🧑‍🏫 Walkthrough.** Gradient checking is the standard safety net when you derive gradients by hand: a sign error or a missing factor of 2 shows up instantly as a mismatch. Deep-learning libraries even ship tools (`gradcheck`) that do exactly this.

### 🧩 Problem 8.4: Write gradient descent &nbsp; ⭐⭐

Implement `gradient_descent(x, y, lr, n_iter)` for the line ŷ = w·x + b with the MSE cost. Start from `w = b = 0` and return `(w, b, history)`, where `history` is a list of the **cost before each update**.

Then run it on data generated from the line **y = 4x + 7** (no noise) with `lr = 0.1` and `n_iter = 500`, storing the result in `w_fit, b_fit, hist`.

In [8]:
def gradient_descent(x, y, lr, n_iter):
    w, b, history = 0.0, 0.0, []
    for _ in range(n_iter):
        err = w * x + b - y
        history.append(np.mean(err ** 2))
        grad_w, grad_b = 2 * np.mean(err * x), 2 * np.mean(err)
        w, b = w - lr * grad_w, b - lr * grad_b
    return w, b, history

x_data = np.linspace(-1, 1, 50)
y_data = 4 * x_data + 7
w_fit, b_fit, hist = gradient_descent(x_data, y_data, lr=0.1, n_iter=500)
print(f"w = {w_fit:.4f}, b = {b_fit:.4f}, final cost = {hist[-1]:.2e}")

w = 4.0000, b = 7.0000, final cost = 5.66e-30


In [9]:
# ✔ Self-check
check("slope recovered (≈ 4)", lambda: abs(w_fit - 4) < 0.01)
check("intercept recovered (≈ 7)", lambda: abs(b_fit - 7) < 0.01)
check("cost decreased at every iteration", lambda: all(np.diff(hist) <= 1e-12))
check("final cost is tiny (< 1e-4)", lambda: hist[-1] < 1e-4)

✅ PASSED   slope recovered (≈ 4)
✅ PASSED   intercept recovered (≈ 7)
✅ PASSED   cost decreased at every iteration
✅ PASSED   final cost is tiny (< 1e-4)


True

**🧑‍🏫 Walkthrough.** Both gradients are computed from the *same* error vector before either parameter changes: a "simultaneous update". (Updating `w` first and then recomputing the error for `b` is a subtle bug that makes the algorithm behave differently.) The cost falls monotonically because the learning rate is small enough for this bowl.

### 🧩 Problem 8.5: Find the best learning rate &nbsp; ⭐⭐

On the standardised student data (provided), run gradient descent (from `w = b = 0`) for each `lr` in `[0.001, 0.01, 0.1, 0.5, 1.1]`, for **at most 1000 iterations**.

For each rate, record the **number of iterations needed** until the cost is within `1e-6` of the *optimal* cost (the optimum is provided as `opt_cost`). If the rate never gets there (too slow, or diverging), record `None`.

Store the results in `iterations` (`{lr: n or None}`) and the rate that needed the fewest iterations in `best_lr`.

In [10]:
students = load("students.csv")
xs = ((students["hours_studied"] - students["hours_studied"].mean()) / students["hours_studied"].std()).to_numpy()
ys = ((students["exam_score"] - students["exam_score"].mean()) / students["exam_score"].std()).to_numpy()
opt_w, opt_b = np.polyfit(xs, ys, 1)
opt_cost = np.mean((opt_w * xs + opt_b - ys) ** 2)

rates = [0.001, 0.01, 0.1, 0.5, 1.1]
iterations = {}
with np.errstate(all="ignore"):                 # silence overflow warnings from the diverging run
    for lr in rates:
        w = b = 0.0
        iterations[lr] = None
        for i in range(1, 1001):
            err = w * xs + b - ys
            if np.mean(err ** 2) < opt_cost + 1e-6:
                iterations[lr] = i
                break
            w, b = w - lr * 2 * np.mean(err * xs), b - lr * 2 * np.mean(err)

best_lr = min((lr for lr in rates if iterations[lr] is not None), key=lambda lr: iterations[lr])
print(iterations, "→ best α =", best_lr)

{0.001: None, 0.01: 340, 0.1: 32, 0.5: 3, 1.1: None} → best α = 0.5


In [11]:
# ✔ Self-check
check("results for all five rates", lambda: sorted(iterations) == sorted(rates))
check("α = 1.1 never converges (diverges)", lambda: iterations[1.1] is None)
check("α = 0.001 is too slow for 1000 iterations", lambda: iterations[0.001] is None)
check("α = 0.01 converges, but needs hundreds of iterations", lambda: iterations[0.01] is not None and iterations[0.01] > 100)
check("α = 0.5 is the fastest", lambda: best_lr == 0.5)

✅ PASSED   results for all five rates
✅ PASSED   α = 1.1 never converges (diverges)
✅ PASSED   α = 0.001 is too slow for 1000 iterations
✅ PASSED   α = 0.01 converges, but needs hundreds of iterations
✅ PASSED   α = 0.5 is the fastest


True

**🧑‍🏫 Walkthrough.** α = 0.001 would need ~3,000+ iterations; α = 0.01 needs ~340; α = 0.1 ~32; α = 0.5 reaches the optimum almost immediately (on perfectly standardised data the bowl is round, so a step of 0.5 lands exactly on the bottom); and α = 1.1 overshoots a little more on every step and diverges. Remember, though, that 0.5 is only optimal for *this* well-conditioned problem: on real data we typically use smaller, safer values.

### 🧩 Problem 8.6: Stop when it's converged &nbsp; ⭐⭐

Write `gd_with_tol(x, y, lr, tol, max_iter=100000)` that runs gradient descent from `w = b = 0` and **stops as soon as the cost decreases by less than `tol`** between iterations. Return `(w, b, n_iter)`.

Run it on standardised student data (provided) with `lr = 0.1` for `tol = 1e-3` and `tol = 1e-8`; store the iteration counts in `n_loose` and `n_tight`.

In [12]:
students = load("students.csv")
xs = ((students["hours_studied"] - students["hours_studied"].mean()) / students["hours_studied"].std()).to_numpy()
ys = ((students["exam_score"] - students["exam_score"].mean()) / students["exam_score"].std()).to_numpy()

def gd_with_tol(x, y, lr, tol, max_iter=100000):
    w = b = 0.0
    prev_cost = np.inf
    for i in range(1, max_iter + 1):
        err = w * x + b - y
        cost = np.mean(err ** 2)
        if abs(prev_cost - cost) < tol:
            return w, b, i
        prev_cost = cost
        w, b = w - lr * 2 * np.mean(err * x), b - lr * 2 * np.mean(err)
    return w, b, max_iter

_, _, n_loose = gd_with_tol(xs, ys, 0.1, 1e-3)
w_tight, b_tight, n_tight = gd_with_tol(xs, ys, 0.1, 1e-8)
print(f"loose tolerance: {n_loose} iterations;  tight tolerance: {n_tight} iterations")

loose tolerance: 15 iterations;  tight tolerance: 41 iterations


In [13]:
# ✔ Self-check
_w, _b = np.polyfit(xs, ys, 1)
check("a tighter tolerance needs more iterations", lambda: n_tight > n_loose > 1)
check("both stop long before max_iter (< 1000)", lambda: n_tight < 1000)
check("the tight run is very close to the exact solution (within 0.001)", lambda: abs(w_tight - _w) < 1e-3 and abs(b_tight - _b) < 1e-3)

✅ PASSED   a tighter tolerance needs more iterations
✅ PASSED   both stop long before max_iter (< 1000)
✅ PASSED   the tight run is very close to the exact solution (within 0.001)


True

**🧑‍🏫 Walkthrough.** The cost improvement shrinks geometrically, so each extra digit of precision costs a roughly constant number of additional iterations. The tolerance is a trade-off between precision and compute, and `max_iter` is the safety net for when the rule never triggers.

### 🧩 Problem 8.7: Logistic regression from scratch &nbsp; ⭐⭐⭐

Implement `fit_logistic_gd(X, y, lr, n_iter)` using **only NumPy**. It starts from zeros and returns `(w, b, losses)`, where `losses` is the log loss *before each update*.

The gradients are `X.T @ (p − y) / n` for `w` and `mean(p − y)` for `b`, with `p = sigmoid(X @ w + b)`.

Train it on the standardised churn data (provided; `lr = 0.5`, `n_iter = 1000`) and store the training accuracy at a 0.5 threshold in `my_acc`.

In [14]:
churn = load("churn.csv")
X = StandardScaler().fit_transform(churn.drop(columns="churned"))
y = churn["churned"].to_numpy()

def fit_logistic_gd(X, y, lr, n_iter):
    sigmoid = lambda z: 1 / (1 + np.exp(-z))
    w, b, losses = np.zeros(X.shape[1]), 0.0, []
    for _ in range(n_iter):
        p = sigmoid(X @ w + b)
        losses.append(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
        w = w - lr * (X.T @ (p - y)) / len(y)
        b = b - lr * np.mean(p - y)
    return w, b, losses

w, b, losses = fit_logistic_gd(X, y, lr=0.5, n_iter=1000)
my_acc = np.mean(((1 / (1 + np.exp(-(X @ w + b)))) >= 0.5).astype(int) == y)
print(f"loss {losses[0]:.3f} → {losses[-1]:.3f};  training accuracy {my_acc:.3f}")

loss 0.693 → 0.337;  training accuracy 0.853


In [15]:
# ✔ Self-check
_ref = LogisticRegression(C=1e10, max_iter=5000).fit(X, y)
check("log loss decreases", lambda: losses[-1] < losses[0] and np.all(np.diff(losses) < 1e-9))
check("weights match scikit-learn (max difference < 0.01)", lambda: np.max(np.abs(w - _ref.coef_[0])) < 0.01)
check("accuracy matches scikit-learn's", lambda: abs(my_acc - _ref.score(X, y)) < 0.005)

✅ PASSED   log loss decreases
✅ PASSED   weights match scikit-learn (max difference < 0.01)
✅ PASSED   accuracy matches scikit-learn's


True

**🧑‍🏫 Walkthrough.** The code is almost identical to linear regression's gradient descent: the only change is that predictions are squashed through the sigmoid, and the loss is log loss rather than squared error. This is the key insight of the lesson: **one optimisation loop trains an enormous variety of models.** (Training accuracy ~85% looks decent, but as Lesson 7 taught us, with 15% churners you should check recall too!)